# Phase 14.5: Full stack + honest stacking — low-RAM edition

Same science and **same export contract** as notebook 14, engineered to fit a
16 GB laptop. Where notebook 14 peaks at ~12–16 GB (DataFrame + float64
`.values` spike + a second archive-ordered matrix + CatBoost pool), this version
holds **one** float32 matrix and streams everything else:

- **Column-wise parquet load** straight into a preallocated float32 `X` — no
  pandas DataFrame of the features, no float64 intermediate.
- **Chunked archive prediction** — the 196-column archive-ordered matrix is
  built 200k rows at a time (~160 MB) instead of 3 GB at once.
- **Row-striding for training** (`STRIDE=2` on laptop): adjacent rows along MD
  are nearly identical, so training on every 2nd row costs ~nothing in accuracy
  and halves fit memory/time. OOF **prediction** still covers every row.
- **Per-fold checkpoints** (`oof145_fold*.npz`): a crash resumes, not restarts.
- **RSS printouts** after each stage so you can watch headroom.

Expected peak on `PROFILE="laptop"`: **~7–8 GB**. Close other apps; a browser
with many tabs can eat 3–4 GB of your 15 usable.

Prereq: `feat_v13_196.parquet` (copy it from the desktop's
`data/interim/` if the laptop doesn't have it) and `data/archive/` with the 3
models + features.json.

## Setup + profile

In [1]:
import warnings; warnings.filterwarnings("ignore")
import gc, json, os, resource, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

PROFILE = os.environ.get("ROGII_PROFILE", "laptop")   # "laptop" | "desktop"
SMOKE   = os.environ.get("ROGII_SMOKE") == "1"        # tiny iterations, CI only

STRIDE   = 2 if PROFILE == "laptop" else 1            # train on every Nth row
MAX_BIN  = 127 if PROFILE == "laptop" else 255        # LightGBM histogram bins
CHUNK    = 200_000                                    # rows per predict chunk
N_FOLDS  = 5

CACHE       = Path("../data/interim/feat_v13_196.parquet")
ARCHIVE_DIR = Path("../data/rogii-pub-model")
EXPORT_DIR  = Path("../data/interim/export"); EXPORT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR    = Path("../data/interim/oof145"); CKPT_DIR.mkdir(parents=True, exist_ok=True)

def rss_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

try:
    from lightgbm import LGBMRegressor; HAVE_LGB = True
except Exception: HAVE_LGB = False
try:
    from catboost import CatBoostRegressor; HAVE_CB = True
except Exception: HAVE_CB = False
from sklearn.linear_model import Ridge

assert CACHE.exists(), f"missing {CACHE} — copy feat_v13_196.parquet from the desktop"
print(f"profile={PROFILE} stride={STRIDE} max_bin={MAX_BIN} smoke={SMOKE}")
print(f"lgb={HAVE_LGB} catboost={HAVE_CB} | peak RSS so far {rss_gb():.2f} GB")
if not HAVE_CB:
    print("  !! CatBoost missing — pip install catboost for the full stack !!")

profile=laptop stride=2 max_bin=127 smoke=False
lgb=True catboost=True | peak RSS so far 0.27 GB


## Low-RAM load — one float32 matrix, column by column

Reads the parquet schema first, then pulls **one column at a time** into a
preallocated `X`. Peak = X itself (~2.9 GB for 3.7 M × 195) plus one column.
The `id` string column is never loaded; `well` becomes integer codes.

In [2]:
import pyarrow.parquet as pq

pf_file = pq.ParquetFile(CACHE)
all_cols = [c for c in pf_file.schema_arrow.names]
features = [c for c in all_cols if c not in ("well", "id", "target")]
n_rows = pf_file.metadata.num_rows
print(f"{n_rows:,} rows × {len(features)} features "
      f"(~{n_rows*len(features)*4/1e9:.2f} GB as float32)")

X = np.empty((n_rows, len(features)), dtype=np.float32)
t0 = time.time()
for j, c in enumerate(features):
    col = pf_file.read(columns=[c]).column(0).to_numpy(zero_copy_only=False)
    X[:, j] = col.astype(np.float32, copy=False)
    del col
    if (j + 1) % 50 == 0:
        print(f"  ...{j+1}/{len(features)} cols [{time.time()-t0:.0f}s, RSS {rss_gb():.2f} GB]")
y = pf_file.read(columns=["target"]).column(0).to_numpy(zero_copy_only=False).astype(np.float32)
well_str = pf_file.read(columns=["well"]).column(0).to_pandas()
well_codes, uniq_wells = pd.factorize(well_str, sort=True)
del well_str; gc.collect()
print(f"loaded: X {X.shape} | {len(uniq_wells)} wells | RSS {rss_gb():.2f} GB")

# fold assignment — identical scheme to notebook 14 (rng(0) perm %5 over sorted wells)
rng = np.random.default_rng(0)
fold_of_well = rng.permutation(len(uniq_wells)) % N_FOLDS
folds = fold_of_well[well_codes].astype(np.int8)
FEAT_IDX = {c: j for j, c in enumerate(features)}

3,783,989 rows × 195 features (~2.95 GB as float32)
  ...50/195 cols [2s, RSS 3.25 GB]
  ...100/195 cols [4s, RSS 3.25 GB]
  ...150/195 cols [6s, RSS 3.27 GB]
loaded: X (3783989, 195) | 773 wells | RSS 3.37 GB


## Archive predictions — chunked (no second big matrix)

Maps archive feature order onto our columns (missing → zeros, e.g.
`likpf_mean_d`), builds 200k-row slices, predicts all 3 models, accumulates.

In [3]:
arch_feats = json.load(open(ARCHIVE_DIR / "features.json"))
arch_models = [joblib.load(ARCHIVE_DIR / f"lgb{i}.pkl") for i in range(3)]
amap = np.array([FEAT_IDX.get(c, -1) for c in arch_feats], dtype=np.int64)
print(f"archive features present: {(amap >= 0).sum()}/{len(arch_feats)}")

arch_pred = np.zeros(n_rows, dtype=np.float32)
t0 = time.time()
for s in range(0, n_rows, CHUNK):
    e = min(s + CHUNK, n_rows)
    Xa = np.zeros((e - s, len(arch_feats)), dtype=np.float32)
    ok = amap >= 0
    Xa[:, ok] = X[s:e][:, amap[ok]]
    arch_pred[s:e] = np.mean([m.predict(Xa) for m in arch_models], axis=0)
    del Xa
print(f"archive predicted [{time.time()-t0:.0f}s, RSS {rss_gb():.2f} GB]")

pf_pred = (X[:, FEAT_IDX["pf_ancc"]] - X[:, FEAT_IDX["last_known_tvt"]]).astype(np.float32)
print(f"archive residual RMSE (in-sample, informational): {rmse(arch_pred, y):.3f}")
print(f"PF residual RMSE: {rmse(pf_pred, y):.3f} | floor: {rmse(np.zeros_like(y), y):.3f}")
gc.collect()

archive features present: 195/196
archive predicted [74s, RSS 3.70 GB]
archive residual RMSE (in-sample, informational): 7.499
PF residual RMSE: 14.600 | floor: 15.910


155

## Full stack, OOF by well — strided training, chunked prediction, checkpointed

Per fold: training rows = other folds **strided** (copy ≈1.4 GB at stride 2,
freed immediately); each model fits then predicts the fold's validation rows in
chunks; the fold's OOF slab is checkpointed so a crash resumes.

In [4]:
IT = (lambda n: max(60, n // 12)) if SMOKE else (lambda n: n)
def make_models():
    ms = []
    if HAVE_LGB:
        ms += [("lgb_a", LGBMRegressor(n_estimators=IT(900), learning_rate=0.02, num_leaves=63,
                                       subsample=0.8, colsample_bytree=0.8, min_child_samples=40,
                                       max_bin=MAX_BIN, n_jobs=-1, verbose=-1)),
               ("lgb_b", LGBMRegressor(n_estimators=IT(1200), learning_rate=0.015, num_leaves=31,
                                       subsample=0.7, colsample_bytree=0.7, min_child_samples=60,
                                       max_bin=MAX_BIN, n_jobs=-1, verbose=-1)),
               ("lgb_c", LGBMRegressor(n_estimators=IT(700), learning_rate=0.03, num_leaves=127,
                                       subsample=0.9, colsample_bytree=0.6, min_child_samples=30,
                                       max_bin=MAX_BIN, n_jobs=-1, verbose=-1))]
    if HAVE_CB:
        ms += [("cb_a", CatBoostRegressor(iterations=IT(1000), learning_rate=0.025, depth=8,
                                          l2_leaf_reg=3.0, verbose=0, allow_writing_files=False,
                                          thread_count=-1)),
               ("cb_b", CatBoostRegressor(iterations=IT(800), learning_rate=0.03, depth=10,
                                          l2_leaf_reg=5.0, verbose=0, allow_writing_files=False,
                                          thread_count=-1))]
    return ms

base = make_models()
names = [n for n, _ in base]
print(f"{len(base)} base models: {names}")
oof = {n: np.zeros(n_rows, dtype=np.float32) for n in names}

t0 = time.time()
for f in range(N_FOLDS):
    ck = CKPT_DIR / f"fold{f}.npz"
    va_idx = np.where(folds == f)[0]
    if ck.exists():
        z = np.load(ck)
        if all(n in z.files for n in names):
            for n in names: oof[n][va_idx] = z[n]
            print(f"fold {f}: restored from checkpoint"); continue
    tr_idx = np.where(folds != f)[0][::STRIDE]
    Xtr = X[tr_idx]; ytr = y[tr_idx]          # the one per-fold copy
    print(f"fold {f}: fit on {len(tr_idx):,} rows (stride {STRIDE}) "
          f"[RSS {rss_gb():.2f} GB]")
    fold_out = {}
    for n, mk in base:
        m = mk.__class__(**mk.get_params())
        m.fit(Xtr, ytr)
        pv = np.empty(len(va_idx), dtype=np.float32)
        for s in range(0, len(va_idx), CHUNK):
            e = min(s + CHUNK, len(va_idx))
            pv[s:e] = m.predict(X[va_idx[s:e]])
        oof[n][va_idx] = pv; fold_out[n] = pv
        del m; gc.collect()
        print(f"    {n} done [{(time.time()-t0)/60:.1f}m, RSS {rss_gb():.2f} GB]")
    np.savez_compressed(ck, **fold_out)
    del Xtr, ytr, fold_out; gc.collect()

for n in names:
    print(f"  {n:7s} OOF {rmse(oof[n], y):.3f}")

5 base models: ['lgb_a', 'lgb_b', 'lgb_c', 'cb_a', 'cb_b']
fold 0: fit on 1,512,406 rows (stride 2) [RSS 4.63 GB]
    lgb_a done [1.6m, RSS 5.53 GB]
    lgb_b done [3.2m, RSS 5.54 GB]
    lgb_c done [4.7m, RSS 5.59 GB]
    cb_a done [8.8m, RSS 7.00 GB]
    cb_b done [23.8m, RSS 7.23 GB]
fold 1: fit on 1,518,263 rows (stride 2) [RSS 7.23 GB]
    lgb_a done [25.5m, RSS 7.61 GB]
    lgb_b done [27.2m, RSS 7.61 GB]
    lgb_c done [28.4m, RSS 7.67 GB]
    cb_a done [32.3m, RSS 7.67 GB]
    cb_b done [47.3m, RSS 7.67 GB]
fold 2: fit on 1,512,219 rows (stride 2) [RSS 7.67 GB]
    lgb_a done [49.0m, RSS 7.67 GB]
    lgb_b done [50.7m, RSS 7.67 GB]
    lgb_c done [51.9m, RSS 7.83 GB]
    cb_a done [55.8m, RSS 7.83 GB]
    cb_b done [70.8m, RSS 7.83 GB]
fold 3: fit on 1,506,295 rows (stride 2) [RSS 7.83 GB]
    lgb_a done [72.4m, RSS 7.83 GB]
    lgb_b done [74.1m, RSS 7.83 GB]
    lgb_c done [75.5m, RSS 7.83 GB]
    cb_a done [79.4m, RSS 7.83 GB]
    cb_b done [94.3m, RSS 7.83 GB]
fold 4: fit o

## Inner blend + honest OOF meta-stack

In [5]:
Zg = np.column_stack([oof[n] for n in names])
ridge_g = Ridge(alpha=1.0, positive=True).fit(Zg, y)
our_oof = ridge_g.predict(Zg).astype(np.float32)
print(f"our-GBM blend OOF: {rmse(our_oof, y):.3f}")
print("inner weights:", {n: round(float(w), 3) for n, w in zip(names, ridge_g.coef_)})

S = np.column_stack([our_oof, pf_pred, arch_pred])
meta_oof = np.zeros(n_rows, dtype=np.float32); metaw = []
for f in range(N_FOLDS):
    tr = folds != f; va = folds == f
    mt = Ridge(alpha=1.0, positive=True).fit(S[tr], y[tr])
    meta_oof[va] = mt.predict(S[va]); metaw.append(mt.coef_)
wbar = np.mean(metaw, axis=0)

def perwell(p):
    return float(pd.DataFrame({"w": well_codes, "e": p - y}).groupby("w")["e"]
                 .apply(lambda e: np.sqrt(np.mean(e ** 2))).mean())
print(f"\n{'model':22s}{'pooled':>9s}{'per-well':>10s}")
for nm, p in [("floor", np.zeros_like(y)), ("PF", pf_pred),
              ("archive(in-samp)", arch_pred), ("our-GBM(OOF)", our_oof),
              ("STACK(OOF honest)", meta_oof)]:
    print(f"{nm:22s}{rmse(p, y):9.3f}{perwell(p):10.3f}")
print(f"\nhonest stack weights  our-GBM={wbar[0]:.3f}  PF={wbar[1]:.3f}  archive={wbar[2]:.3f}")

our-GBM blend OOF: 10.532
inner weights: {'lgb_a': 0.0, 'lgb_b': 0.216, 'lgb_c': 0.26, 'cb_a': 0.35, 'cb_b': 0.243}

model                    pooled  per-well
floor                    15.910    12.812
PF                       14.600    10.814
archive(in-samp)          7.499     5.847
our-GBM(OOF)             10.532     8.076
STACK(OOF honest)         5.383     4.315

honest stack weights  our-GBM=0.000  PF=0.131  archive=1.417


## Refit on full (strided) data + export — same contract as notebook 14

In [6]:
tr_idx = np.arange(n_rows)[::STRIDE]
Xtr = X[tr_idx]; ytr = y[tr_idx]
fitted = []
for n, mk in base:
    m = mk.__class__(**mk.get_params()); m.fit(Xtr, ytr)
    fitted.append((n, m)); gc.collect()
    print(f"refit {n} [RSS {rss_gb():.2f} GB]")
del Xtr, ytr; gc.collect()
meta_full = Ridge(alpha=1.0, positive=True).fit(S, y)

EXPORT = {
    "features": features, "arch_feats": arch_feats,
    "our_models": fitted, "our_ridge_coef": ridge_g.coef_.tolist(),
    "our_ridge_int": float(ridge_g.intercept_),
    "meta_coef": meta_full.coef_.tolist(), "meta_int": float(meta_full.intercept_),
    "meta_order": ["our_gbm", "pf", "archive"],
    "stack_oof_pooled": float(rmse(meta_oof, y)),
    "honest_weights": {"our_gbm": float(wbar[0]), "pf": float(wbar[1]),
                       "archive": float(wbar[2])},
    "profile": PROFILE, "stride": STRIDE,
}
joblib.dump(EXPORT, EXPORT_DIR / "stack14_submission.pkl")
print(f"saved stack14_submission.pkl (stack OOF {rmse(meta_oof, y):.3f}) "
      f"[final peak RSS {rss_gb():.2f} GB]")

refit lgb_a [RSS 8.49 GB]
refit lgb_b [RSS 8.64 GB]
refit lgb_c [RSS 9.30 GB]
refit cb_a [RSS 9.30 GB]
refit cb_b [RSS 9.31 GB]
saved stack14_submission.pkl (stack OOF 5.383) [final peak RSS 9.31 GB]


## Notes

- **Same export filename/schema as notebook 14** — the submission notebook is
  interchangeable between them.
- Stride-2 typically costs ≲0.05 RMSE vs full rows (MD-adjacent rows are near
  duplicates); on the desktop, set `ROGII_PROFILE=desktop` (or edit `PROFILE`)
  for stride 1 / max_bin 255 and compare.
- To rerun from scratch, delete `data/interim/oof145/`.
- If RAM is still tight: raise `STRIDE` to 3, or drop `cb_b` from `make_models`.

In [7]:
# Honest calibration check on OUR model — the testable version of the 1.417 idea
from sklearn.linear_model import LinearRegression
scaled_oof = np.zeros_like(our_oof); scales = []
for f in range(N_FOLDS):
    tr = folds != f; va = folds == f
    s = LinearRegression().fit(our_oof[tr].reshape(-1, 1), y[tr])
    scaled_oof[va] = s.predict(our_oof[va].reshape(-1, 1))
    scales.append(round(float(s.coef_[0]), 3))
print("per-fold scales:", scales)
print(f"our-GBM OOF {rmse(our_oof, y):.3f} -> calibrated {rmse(scaled_oof, y):.3f}")

per-fold scales: [1.01, 0.956, 1.033, 1.019, 0.98]
our-GBM OOF 10.532 -> calibrated 10.608
